# So far:

- Given p, q, and e, I can calculate n and d

- I'm happy with functionality, names, docstrings

- Speed is ???

# To Do:

- Make it actually interactive (i.e. encode a message!)

  - How to choose primes?

  - Decide good sized numbers

  - Plaintext into ASCII

     - Should I just define a class?  This seems easiest
  
  - Encrypt --> ASCII into plaintext

     - use emojis for control characters

  - Decrypt

- Decide how to distribute it

- Deal with details of distributing it

In [1]:
from sympy import primerange, nextprime
from math import lcm
import numpy as np
from itertools import batched
import project_path
from rsatools.RNG import *

## Choosing numbers

So ASCII has 128 characters, i.e. each character is 2 hexadecimal digits i.e. 8 bits.  There are 33 control characters and 95 text characters.  I can find 33 emojis to represent the control characters, which should be okay.  (implementation will be *insane* but I'll cross that bridge when I get to it). 

So for a message that is n characters, I'll need 8n bits to encode it.  `hello, world` is 12 characters so I think 16 is a reasonable limit for message length.  If the numbers start to get too insane I'll break it up into 2 words of 8 characters.  16*8 is 128 bits (just like ascii lol) so I need to work with a modulus that is 128 bits long.

For numbers $p$ and $q$ with bitlength $j$ and $k$ respectively we expect that the bitlength of $pq$ is between $j+k-1$ and $j+k$.  This can be seen by taking $p=2^j=1000...$ and $q=2^k=1000...$.  Then the bitlength of $pq$ is clearly $j+k-1$.  

If instead $p=2^{j+1}-1=111...$ and $q=2^{k+1}-1=111...$ then the bitlength of $pq$ must be one less than the bitlength of $(p+1)(q+1)$ (except possibly in extreme cases of $p$ and $q$, which I don't care to think about) i.e. the bitlength of $pq$ is $(j+1)+(k+1)-1-1=j+k$.

Then we should choose primes that are about 65 bits long.  If we choose two 65 bit primes, the bitlength of $pq$ is guaranteed to be either 128 or 129 bits.  

So here's my main question: what is the bitlength of the traditional choice of $e$?  with $e=65537=2^{16}+1$ we must have a bit length of 17.  But that doesn't actually tell us about the size of the primes.  Well, we do know that $1<e<\lambda(pq)$ which might tell us something about the primes themselves.  But really it still only sets a lower bound (and a fuzzy one at that because $\lambda(n)$ isn't as well behaved as $\phi(n)$, from my perspective). 

Okay, instead, let's examine actual implementations of RSA.  According to wikipedia, the key length (that's the bitlength of $n$) is typically between 2048 and 4096 bits.  That means a single key can encrypt like, 256-512 ascii characters.  That's pretty good.

But at the same time, it's hard to write out a 2048 bit integer on the screen.  If $p$ is 2048 bits that implies that $\log_2(p)\approx2048$.  It follows that $$\log_{10}(p)=\frac{\log_2(p)}{\log_2(10)}\approx\frac{2048}{4}\approx512$$ so we conclude that such a prime is around 512 digits long.  

By contrast, if $p$ or $n$ is 64 bits long, its only about 16 digits long.  That's a big number but still much more managable than a 512 digit prime.  Although Victor was talking about his experience using 15 digit primes and being unable to crack them.  To be fair, I'm using the cryptosystem in the forward direction and he was trying to use it in the reverse direction, which gives me a significant advantage.  

My main concern is that for some primes in this range, we will have $\lambda(pq)<e=65537$.  I should probably check that (perhaps by seeing if there is a lower bound for $\lambda(n)$)

After some work in Google sheets, I've decided that calculating a lower bound for Carmichael's Totient is not very fun.  At least not from the perspective of finding n such that $\lambda(n)>x$ for some given x.  For example, to guarantee $\lambda(n)>36$, you have to have $n>138\,181\,680$ which is already insane.  Considering I wanted to ensure $\lambda(n)>65537$, that is infeasable.  

I couuuuuuuuuld just pick $e=17$.  Then I only need $n>65520$ which means I can use quite small primes if I want to.  17 is the last Fermat Prime before 257, which I haven't calculated the necessary n for.  

Okay so I've got a good candidate.  With $e=257$, we need to keep $n>50\,365\,204\,798\,708\,800$.  That's not horrible.  That ensures that $\lambda(n)>e$.  If you're curious, $\lambda(50\,365\,204\,798\,708\,800)=240$.  

So then what must our key length be? how many bits does it take to represent 50 quadrillion? we can find that by rounding up to the next power of 2.  

In [2]:
print(2**56)

72057594037927936


In [3]:
print(2**56 // 1_000_000_000_000_000)

72


So our key length would be 56 bits at a minimum.  That means (since we were aiming for a key length of ~128 bits anyway) we can choose $e=257$ and not worry about $\lambda(n)<e$

I'm not 100% sure, but I'm guessing that our primes probably shouldnt be anywhere less than 56 bits either.  So maybe we'll select two primes between 2^65 and 2^66.  At a minimum, that allows us to encrypt 16 characters.  

So how do we go about implementing this?  Well, keeping e fixed at 257 means that we are only interested in finding the multiplicative inverse of 257 modulo pq.  So there's not much need for a general `modinverse()` function.  We can create a more specific `inverse257()` function which takes a single argument (n) and gives a single output (d). This is very appealing, because there may be improvements we can make to the algorithm to make things faster.  This is not strictly necessary, but its nice to have the option to refactor the code in this way in the future.  Let's copy in some of the past work we've done.  

In [4]:
# from math import lcm

# I moved all the imports to the top of the notebook for convenience

def carmichael(p: int, q: int) -> int:
    """Return Carmichael's totient of p*q, only if p and q are prime.
    
    This function returns the least common multiple of p-1 and q-1. 
    If p and q are prime, this is Carmichael's totient of their 
    product.  This is helpful for the RSA cryptosystem.  
    """
    return lcm(p-1, q-1)

In [5]:
def modinverse(a: int, n: int) -> int:
    """Return the multiplicative inverse of `a` modulo `n`.
    
    This function uses the extended Euclidean algorithm to solve 
    for `gcd(a,n)` and minimal Bezout Coefficients (BC). Assuming 
    that `gcd(a,n)=1`, the positive minimal BC of `a` is exactly the
    multiplicative inverse of `a` mod `n`. After the `while()` loop, 
    `r0 = gcd(a,n)`, `s0` is a minimal BC for `a`, and `t0` is the 
    relevant BC for `n`.  The sign of the inputs is not used, so `a` 
    and `-a` give the same output.  `modinverse(a, b)` will always 
    output a positive number.  Using 0 as either input will probably
    break something.
    """

    if abs(a)==abs(n):
        return 1        # this is the trivial case of a=b
                        #
    elif abs(a)>abs(n): # WANT: sa + tb = gcd(a,b) AND r0=a
        r0 = abs(a)
        r1 = abs(n)
        s0 = 1
        t0 = 0
        s1 = 0
        t1 = 1
                        #
    elif abs(a)<abs(n): # WANT: sa + tb = gcd(a,b) AND r0=b
        r0 = abs(n)     # to achieve this, we need to flip
        r1 = abs(a)     # the definitions of s and t
        s0 = 0
        t0 = 1
        s1 = 1
        t1 = 0

    else:
        raise RuntimeError("This is impossible.")
                        #
    while not r1==0:    # Apply the extended Euclidean alg.
        q1 = r0 // r1
        r2 = r0 % r1
        s2 = s0 - (q1*s1)
        t2 = t0 - (q1*t1)
        r0 = r1
        r1 = r2
        s0 = s1
        s1 = s2
        t0 = t1
        t1 = t2
                        #
    if s0>0:            # now ensure positive output
        return s0  
        # s is already positive

    elif s0<0:
        return s0+int(abs(n)/r0)
        # we find the *other* minimal bezout coefficients
        # so that s is positive and t is negative

    else:
        raise RuntimeError("This should never happen.")

In [6]:
def inverse257(n: int) -> int:
    """Find the multiplicative inverse of 257 modulo n and return an int.
    
    This function uses the extended Euclidean Algorithm to find minimal 
    coefficients to Bezout's Identity, i.e. find s and t such that
    `s*257 + t*n = 1`. The minimal positive value of s is exactly the
    multiplicative inverse of 257 mod n. 

    Generally, inputs smaller than `n=50_365_204_798_708_800` are not 
    reccommended, although they should work as normal. Inputs smaller
    than `n=65520` are not allowed and will raise `RuntimeError`.  
    """

    if n<=65520 or type(n) is not int:
        raise RuntimeError(
            "inverse257() only accepts integer inputs larger than 65520."
        )
    else:
        pass

    r0 = 257
    r1 = n    # all integers greater than 65520 are positive
    s0 = 1
    t0 = 0
    s1 = 0
    t1 = 1
                        #
    while not r1==0:    # Apply the extended Euclidean alg.
        q1 = r0 // r1
        r2 = r0 % r1
        s2 = s0 - (q1*s1)
        t2 = t0 - (q1*t1)
        r0 = r1
        r1 = r2
        s0 = s1
        s1 = s2
        t0 = t1
        t1 = t2
                        #
    if s0>0:            # now ensure positive output
        return int(s0)  
        # s is already positive

    elif s0<0:
        return int(s0+int(abs(n)/r0))
        # we find the *other* minimal bezout coefficients
        # so that s is positive and t is negative

    else:
        raise RuntimeError("This should never happen.")

In [7]:
print(str(inverse257(65521)))
print(str((inverse257(65521)*257)%65521))

51499
1


Great! This simplifies that function quite a bit.  There may be ways to speed up the extended Euclidean Algorithm since we've fixed e=257, but that does not need to be done right now.  More importantly, we need to start working on how primes are chosen.  

In the paper draft, Victor used the function `randprime()` from the `sympy` package.  I've never heard of sympy but it seems like a good candidate.  Unless numpy has a better random prime function, I'll probably copy what Victor used.

Numpy does not have a random prime generator.  While looking through the docs, though, I remembered why I'm fond of numpy's other RNG capabilities.  To get random numbers from numpy, we first create a "Generator" object.  Then we can call methods of that object, which will return random values according to whatever method was used.  Here's what I like: when we create the generator, we have the option to include a seed for the generator.  Two generators using the same seed will act in the same way.  This means that every time we run the code, we get the same outputs-which is very useful for debugging.  When we want to distribute the code, we can just remove the seed (I think numpy then uses the unix time as the seed).  

So I'd very much appreciate it if sympy also had this functionality.  We will check.  

I'm really not seeing any options in the documentation to provide a seed to the `randprime()` function.  *(Oddly, when I perform a Google search, Gemini is convinced that sympy accepts seeds for RNG functions.  Stupid clanker.)*  Also it mentions that the primes are not actually random.  So I will try to come up with a compromise.  But first I need to test how good the `primerange()` function from sympy.  

Of course, I'll need to install the package... idiot...

In [8]:
# from sympy import primerange

# I moved all the imports to the top of the notebook for convenience.

In [9]:
print(list(primerange(0, 100)))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]


In [10]:
primes_to_256 = primerange(2**0, 2**8)

prime_generator_TEST = batched(primes_to_256, 10)

# the structure of the while loop below is copied from Lucas Resende on StackOverflow

while True:
    try:
        npgTEST = next(prime_generator_TEST)
        print("".join([f"{x:3.0f}"+"  " for x in npgTEST]))

        # its horrible but its python

    except StopIteration:
        print("Look Mom I caught an exception!")
        break

  2    3    5    7   11   13   17   19   23   29  
 31   37   41   43   47   53   59   61   67   71  
 73   79   83   89   97  101  103  107  109  113  
127  131  137  139  149  151  157  163  167  173  
179  181  191  193  197  199  211  223  227  229  
233  239  241  251  
Look Mom I caught an exception!


Very nice.  How does it do with a larger range?

In [11]:
BFAoP = np.array(list(primerange(2**24, 2**25)))
print(str(len(BFAoP)))

985818


Okay this is not going to work.  That took 17 seconds and thats orders of magnitude smaller than I need for the actual RSA implementation.  Which really sucks.  

Okay, but inspecting the source code of `sympy.randprime()`, we can see that there is an easy compromise.  Internally, `sympy.randprime()` generates a random integer, then finds the next prime number.  That's a smart way to do it and we can use numpy's random generator with a seed to generate the random integer.  

Since sympy is licenced under a BSD 3 clause licence, and the main licence on this repository is GPL v2, I will break that code out into a separate .py file.  I'll put that in a different folder, for licenceing clarity.  

Okay I've got a dodgy version of the random generation implemented.  I'm happy with how everythin is named, so in theory this notebook will continue to work even if I update the implementation more.  

In [12]:
rng1 = prime_rng(137)

In [13]:
rng1.randprime(2**24, 2**25)

25017127

Great! the cell above should always output 25017127.  That will work well.

So we have random prime generation with a seed, which is exceptionally helpful.  This makes debugging easier and means that any demonstration can be guaranteed to work.  Not sure it was the best idea to do battle with Python's import system.  That's okay though.  

Now we have to deal with encryption and decryption. That means dealing with ASCII interpretation. Which is going to be a pain.  

Actually, first, lets check that some of this stuff works in a reasonable amount of time.  I'll use an online calculator for the ascii conversion and we'll check that the RSA algorithm works so far.

hello, world = `68 65 6c 6c 6f 2c 20 77 6f 72 6c 64` (hex)

In [14]:
message = int("68656c6c6f2c20776f726c64", base=16)
print(message)
print(len(str(message)))

32309054545040288110865443940
29


In [15]:
rng = prime_rng(8675309)
p = rng.randprime(2**62, 2**63)
q = rng.randprime(2**62, 2**63)
n = p*q
print(p, q, p*q, hex(p*q), len(str(hex(p*q))), sep="\n")

6359204061147409477
8367953916027854419
53213526526298700520743363496336928863
0x28088d6b49e28bc92726c9d5acc7b05f
34


so with those two primes, we get 34/2 = 17 characters in any encrypted message.  For some reason, the randprime function throws an error whenever I put the bounds any higher than this.  That's annoying, but 17 characters is pretty good.  

In [16]:
lamb = carmichael(p, q)
print(lamb, len(str(hex(lamb))), sep="\n")

8868921087716450084336034253193610828
33


MUCH larger than 257.  so we are chillin with our choice of e

In [17]:
private_key = inverse257(lamb)
d = private_key
print(d, d*257, (d*257)%lamb, len(str(hex(d))), sep="\n")

1449395664140431531292270189237866361
372494685684090903542113438634131654777
1
33


Okay, in theory, this works great.  

In [18]:
ciphertext = (message ** 257) % n
print(ciphertext)

27377463826080814984093499563840440554


In [19]:
print(hex(message))
print(hex(ciphertext))

0x68656c6c6f2c20776f726c64
0x1498b52e580de2a0a2e1eceb8a7d08ea


problem- that's an odd number of hex digits for the ciphertext.  That's not necessarily a bad thing, but it will make demonstrating much harder.  We'll have to 1) pad the messages and 2) pad the ciphertext.

In [ ]:
print((ciphertext ** private_key) % n)
print(message)

AAAAAAAAAAAAAAAAAAAAAAAAAAND it never finished.  goddamnit.  The numbers are pretty big- do you think I'll have to implement modular multiplication manually? Is my namespace possibly just too messed up right now?? is 257 too small of a number? is 17 characters simply too much to deal with?  who fucking knows.  I'm going to reorganize this repo a little bit.  

# This is the End of work on this notebook.